# Análise clustering. Barragem de Alqueva

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho_v = ortho_v[(ortho_v['northing'] >= norte_min) & (ortho_v['northing'] <= norte_max) &
                  (ortho_v['easting'] >= este_min) & (ortho_v['easting'] <= este_max)]
ortho_h = ortho_h[(ortho_h['northing'] >= norte_min) & (ortho_h['northing'] <= norte_max) &
                  (ortho_h['easting'] >= este_min) & (ortho_h['easting'] <= este_max)]

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # Ajustar conforme as colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 10. Usar TODAS as células com dados
# ==============================
selected_ids = agg['cell_id'].unique().tolist()
agg_sel = agg.copy()
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids)]



# ==============================
# 11. ORTHO em formato longo e atribuição de células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme datas ORTHO
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

# ==============================
# Transformar ORTHO em formato longo
# ==============================
ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

ortho_v_sel = ortho_v_long.copy()
ortho_h_sel = ortho_h_long.copy()


# atribuir células
for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids)]

# ==============================
# 12. Mapa das células selecionadas + pontos ORTHO (com legenda organizada)
# ==============================

# Criar figura
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, 
                   label=f'Grelha Base ({grid_size} m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, 
                       label=f'Células Selecionadas ({grid_size} m)')

# Pontos ASC/DESC (centros das células)
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Criar GeoDataFrame combinado dos pontos ORTHO selecionados
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)

# Evitar duplicados exatos
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing', 'date'])

# Transformar em GeoDataFrame
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)


# Pontos ORTHO combinados (preto)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Adicionar basemap e formato
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax_map.set_axis_off()

# Legenda organizada (inclui o tamanho da grelha)
ax_map.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()


# ==============================
# 13. Grid de gráficos - TODAS AS CÉLULAS
# ==============================

# Usar todas as células disponíveis dentro dos limites
agg_all = agg.copy()

# Calcular limites globais (min e max de todos os dados)
v_min = min(
    agg_all['dV'].min(),
    agg_all['dH'].min(),
    ortho_v_long['disp'].min() if not ortho_v_long.empty else np.inf,
    ortho_h_long['disp'].min() if not ortho_h_long.empty else np.inf
)
v_max = max(
    agg_all['dV'].max(),
    agg_all['dH'].max(),
    ortho_v_long['disp'].max() if not ortho_v_long.empty else -np.inf,
    ortho_h_long['disp'].max() if not ortho_h_long.empty else -np.inf
)

# Todas as células (ordenadas por Y e X)
all_ids = sorted(agg_all['cell_id'].unique(), key=lambda s: (int(s.split('_')[1]), int(s.split('_')[0])))

# Número de células
n_cells = len(all_ids)
n_cols = 4  # Ajusta se quiseres mais/menos colunas
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*4), sharex=True, sharey=True)

if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(all_ids):
    r = i // n_cols
    c = i % n_cols
    ax = axes[r, c]

    agg_cell = agg_all[agg_all['cell_id'] == cid]
    ortho_v_cell = ortho_v_long[ortho_v_long['cell_id'] == cid]
    ortho_h_cell = ortho_h_long[ortho_h_long['cell_id'] == cid]

    # ASC/DESC
    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')

    # ORTHO (tracejado)
    if not ortho_v_cell.empty:
        ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty:
        ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')

    ax.set_title(f"Célula {cid}", fontsize=9)
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_xlabel("Data")
    ax.set_ylim(v_min, v_max)
    ax.legend(fontsize=7, loc='best')

# Remove eixos vazios
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

#plt.suptitle("Séries Temporais - Todas as Células na Área de Interesse", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho_v = ortho_v[(ortho_v['northing'] >= norte_min) & (ortho_v['northing'] <= norte_max) &
                  (ortho_v['easting'] >= este_min) & (ortho_v['easting'] <= este_max)]
ortho_h = ortho_h[(ortho_h['northing'] >= norte_min) & (ortho_h['northing'] <= norte_max) &
                  (ortho_h['easting'] >= este_min) & (ortho_h['easting'] <= este_max)]

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # Ajustar conforme as colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 10. Usar TODAS as células com dados
# ==============================
selected_ids = agg['cell_id'].unique().tolist()
agg_sel = agg.copy()
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids)]



# ==============================
# 11. ORTHO em formato longo e atribuição de células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme datas ORTHO
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

# ==============================
# Transformar ORTHO em formato longo
# ==============================
ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

ortho_v_sel = ortho_v_long.copy()
ortho_h_sel = ortho_h_long.copy()


# atribuir células
for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids)]

# ==============================
# 12. Mapa das células selecionadas + pontos ORTHO (com legenda organizada)
# ==============================

# Criar figura
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, 
                   label=f'Grelha Base ({grid_size} m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, 
                       label=f'Células Selecionadas ({grid_size} m)')

# Pontos ASC/DESC (centros das células)
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Criar GeoDataFrame combinado dos pontos ORTHO selecionados
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)

# Evitar duplicados exatos
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing', 'date'])

# Transformar em GeoDataFrame
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)


# Pontos ORTHO combinados (preto)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Adicionar basemap e formato
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax_map.set_axis_off()

# Legenda organizada (inclui o tamanho da grelha)
ax_map.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()


# ==============================
# 13. Grid de gráficos - TODAS AS CÉLULAS
# ==============================

# Usar todas as células disponíveis dentro dos limites
agg_all = agg.copy()

# Calcular limites globais (min e max de todos os dados)
v_min = min(
    agg_all['dV'].min(),
    agg_all['dH'].min(),
    asc_long['disp'].min() if not asc_long.empty else np.inf,
    desc_long['disp'].min() if not desc_long.empty else np.inf,
    ortho_v_long['disp'].min() if not ortho_v_long.empty else np.inf,
    ortho_h_long['disp'].min() if not ortho_h_long.empty else np.inf
)
v_max = max(
    agg_all['dV'].max(),
    agg_all['dH'].max(),
    asc_long['disp'].max() if not asc_long.empty else -np.inf,
    desc_long['disp'].max() if not desc_long.empty else -np.inf,
    ortho_v_long['disp'].max() if not ortho_v_long.empty else -np.inf,
    ortho_h_long['disp'].max() if not ortho_h_long.empty else -np.inf
)

# Todas as células (ordenadas por Y e X)
all_ids = sorted(agg_all['cell_id'].unique(), key=lambda s: (int(s.split('_')[1]), int(s.split('_')[0])))

# Número de células e configuração da figura
n_cells = len(all_ids)
n_cols = 4  # ajustar conforme desejado
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*4), sharex=True, sharey=True)
if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(all_ids):
    r = i // n_cols
    c = i % n_cols
    ax = axes[r, c]

    agg_cell = agg_all[agg_all['cell_id'] == cid]
    ortho_v_cell = ortho_v_long[ortho_v_long['cell_id'] == cid]
    ortho_h_cell = ortho_h_long[ortho_h_long['cell_id'] == cid]

    # ASC/DESC combinados (IDW)
    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV IDW')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH IDW')

    # ASC LOS
    asc_cell = asc_long.copy()
    asc_cell['cell_x'] = pd.cut(asc_cell['easting'], bins=x_edges_shifted, labels=False)
    asc_cell['cell_y'] = pd.cut(asc_cell['northing'], bins=y_edges_shifted, labels=False)
    asc_cell.dropna(subset=['cell_x','cell_y'], inplace=True)
    asc_cell['cell_id'] = asc_cell['cell_x'].astype(int).astype(str) + "_" + asc_cell['cell_y'].astype(int).astype(str)
    asc_cell = asc_cell[asc_cell['cell_id'] == cid]
    if not asc_cell.empty:
        ax.plot(asc_cell['date'], asc_cell['disp'], color='cyan', linestyle='-', label='ASC LOS')

    # DESC LOS
    desc_cell = desc_long.copy()
    desc_cell['cell_x'] = pd.cut(desc_cell['easting'], bins=x_edges_shifted, labels=False)
    desc_cell['cell_y'] = pd.cut(desc_cell['northing'], bins=y_edges_shifted, labels=False)
    desc_cell.dropna(subset=['cell_x','cell_y'], inplace=True)
    desc_cell['cell_id'] = desc_cell['cell_x'].astype(int).astype(str) + "_" + desc_cell['cell_y'].astype(int).astype(str)
    desc_cell = desc_cell[desc_cell['cell_id'] == cid]
    if not desc_cell.empty:
        ax.plot(desc_cell['date'], desc_cell['disp'], color='magenta', linestyle='-', label='DESC LOS')

    # ORTHO (tracejado)
    if not ortho_v_cell.empty:
        ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty:
        ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')

    ax.set_title(f"Célula {cid}", fontsize=9)
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_xlabel("Data")
    ax.set_ylim(v_min, v_max)  # mantém mesma escala para todos
    ax.legend(fontsize=7, loc='best')

# Remover eixos vazios
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

plt.tight_layout()
plt.show()



# Clustering

## Temperatura

### Hierárquico (ligação média) + DTW, k=auto=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais para dV
agg_pivot_dv = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot_dv.index.tolist()
X = agg_pivot_dv.values.astype(float)
n = X.shape[0]


# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Grelha: {grid_size} m x {grid_size} m',
#     f'Clustering: DTW + Hierárquico (ligação média)',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)

ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais médias por cluster + temperatura ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_dv.loc[cluster_cells]

    # Séries individuais do cluster (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário, preta)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

from statsmodels.tsa.seasonal import STL

# ==============================
# 13. Decomposição STL das séries médias por cluster
# ==============================

# Ordenar clusters: cluster 1 vem sempre primeiro
clusters_ordered = sorted(clusters_present)
if 1 in clusters_ordered:
    clusters_ordered.remove(1)
    clusters_ordered = [1] + clusters_ordered  # cluster 1 na primeira coluna

fig_stl, axes = plt.subplots(3, n_clusters, figsize=(5 * n_clusters, 10), sharex=True)

# Garantir formato correto se houver apenas um cluster
if n_clusters == 1:
    axes = np.array([axes]).T

# Calcular decomposições STL e limites globais
stl_results = {}
trend_min, trend_max = np.inf, -np.inf
season_min, season_max = np.inf, -np.inf
resid_min, resid_max = np.inf, -np.inf

for cluster_id in clusters_ordered:
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_dv.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)

    ts = pd.Series(cluster_mean_dV.values, index=pd.to_datetime(cluster_mean_dV.index)).asfreq('MS').interpolate()
    stl = STL(ts, period=12, robust=True)
    res = stl.fit()
    stl_results[cluster_id] = res

    # Atualizar limites globais para manter mesma escala entre clusters
    trend_min = min(trend_min, res.trend.min())
    trend_max = max(trend_max, res.trend.max())
    season_min = min(season_min, res.seasonal.min())
    season_max = max(season_max, res.seasonal.max())
    resid_min = min(resid_min, res.resid.min())
    resid_max = max(resid_max, res.resid.max())

# Plot de cada cluster (mesma cor por cluster)
for col_idx, cluster_id in enumerate(clusters_ordered):
    res = stl_results[cluster_id]
    cluster_color = cluster_colors[cluster_id]

    # Tendência
    ax_trend = axes[0, col_idx]
    ax_trend.plot(res.trend, color=cluster_color, linewidth=2.5)
    ax_trend.set_title(f"Tendência – Cluster {cluster_id}", fontsize=12)
    ax_trend.set_ylim(trend_min, trend_max)
    if col_idx == 0:
        ax_trend.set_ylabel("dV (mm)")

    # Sazonal
    ax_seasonal = axes[1, col_idx]
    ax_seasonal.plot(res.seasonal, color=cluster_color, linewidth=2.0)
    ax_seasonal.set_title(f"Sazonal – Cluster {cluster_id}", fontsize=12)
    ax_seasonal.set_ylim(season_min, season_max)
    if col_idx == 0:
        ax_seasonal.set_ylabel("dV (mm)")

    # Resíduo
    ax_resid = axes[2, col_idx]
    ax_resid.plot(res.resid, color=cluster_color, linewidth=1.8)
    ax_resid.set_title(f"Resíduo – Cluster {cluster_id}", fontsize=12)
    ax_resid.set_ylim(resid_min, resid_max)
    if col_idx == 0:
        ax_resid.set_ylabel("dV (mm)")

for ax in axes.flatten():
    ax.grid(False)

plt.suptitle("Decomposição STL das Séries Médias por Cluster (dV)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais para dH
agg_pivot_dh = agg.pivot(index='cell_id', columns='date', values='dH').fillna(0)
cell_ids = agg_pivot_dh.index.tolist()
X = agg_pivot_dh.values.astype(float)
n = X.shape[0]


# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (para dH)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Pivot dH
agg_pivot_dh = agg.pivot(index='cell_id', columns='date', values='dH').fillna(0)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters ---
ax_map = fig.add_subplot(gs[0, :])

grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}'))

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento horizontal (dH).\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries médias por cluster + temperatura ---
dH_min = agg['dH'].min()
dH_max = agg['dH'].max()
dH_margin = (dH_max - dH_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_dh.loc[cluster_cells]

    # Séries individuais (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dH = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dH, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dH_max - dH_min) * 0.25 + (dH_max - (dH_max - dH_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Eixos e limites
    ax.set_ylim(dH_min - dH_margin, dH_max + dH_margin)
    ax2.set_ylim(dH_min - dH_margin, dH_max + dH_margin)

    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dH_max - dH_min) * 0.25 + (dH_max - (dH_max - dH_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dH (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


# ==============================
# 13. Decomposição STL das séries médias por cluster (dH)
# ==============================
from statsmodels.tsa.seasonal import STL

clusters_ordered = sorted(clusters_present)
if 1 in clusters_ordered:
    clusters_ordered.remove(1)
    clusters_ordered = [1] + clusters_ordered

fig_stl, axes = plt.subplots(3, n_clusters, figsize=(5 * n_clusters, 10), sharex=True)
if n_clusters == 1:
    axes = np.array([axes]).T

stl_results = {}
trend_min, trend_max = np.inf, -np.inf
season_min, season_max = np.inf, -np.inf
resid_min, resid_max = np.inf, -np.inf

for cluster_id in clusters_ordered:
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_dh.loc[cluster_cells]
    cluster_mean_dH = cluster_data.mean(axis=0)

    ts = pd.Series(cluster_mean_dH.values, index=pd.to_datetime(cluster_mean_dH.index)).asfreq('MS').interpolate()
    stl = STL(ts, period=12, robust=True)
    res = stl.fit()
    stl_results[cluster_id] = res

    trend_min = min(trend_min, res.trend.min())
    trend_max = max(trend_max, res.trend.max())
    season_min = min(season_min, res.seasonal.min())
    season_max = max(season_max, res.seasonal.max())
    resid_min = min(resid_min, res.resid.min())
    resid_max = max(resid_max, res.resid.max())

for col_idx, cluster_id in enumerate(clusters_ordered):
    res = stl_results[cluster_id]
    cluster_color = cluster_colors[cluster_id]

    ax_trend = axes[0, col_idx]
    ax_trend.plot(res.trend, color=cluster_color, linewidth=2.5)
    ax_trend.set_title(f"Tendência – Cluster {cluster_id}", fontsize=12)
    ax_trend.set_ylim(trend_min, trend_max)
    if col_idx == 0:
        ax_trend.set_ylabel("dH (mm)")

    ax_seasonal = axes[1, col_idx]
    ax_seasonal.plot(res.seasonal, color=cluster_color, linewidth=2.0)
    ax_seasonal.set_title(f"Sazonal – Cluster {cluster_id}", fontsize=12)
    ax_seasonal.set_ylim(season_min, season_max)
    if col_idx == 0:
        ax_seasonal.set_ylabel("dH (mm)")

    ax_resid = axes[2, col_idx]
    ax_resid.plot(res.resid, color=cluster_color, linewidth=1.8)
    ax_resid.set_title(f"Resíduo – Cluster {cluster_id}", fontsize=12)
    ax_resid.set_ylim(resid_min, resid_max)
    if col_idx == 0:
        ax_resid.set_ylabel("dH (mm)")

for ax in axes.flatten():
    ax.grid(False)

plt.suptitle("Decomposição STL das Séries Médias por Cluster (dH)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()



In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Grelha: {grid_size} m x {grid_size} m',
#     f'Clustering: DTW + Hierárquico (ligação média)',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)

ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais médias por cluster + temperatura ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais do cluster (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário, preta)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

from statsmodels.tsa.seasonal import STL

# ==============================
# 13. Decomposição STL comparativa (dV vs dH)
# ==============================

# Clusters ordenados: cluster 1 primeiro
clusters_ordered = sorted(clusters_present)
if 1 in clusters_ordered:
    clusters_ordered.remove(1)
    clusters_ordered = [1] + clusters_ordered

# Configuração da figura: 3 linhas (tendência/sazonal/resíduo), 2*n_clusters colunas (dV e dH lado a lado)
fig_stl, axes = plt.subplots(3, 2 * n_clusters, figsize=(5 * 2 * n_clusters, 10), sharex=True)

if n_clusters == 1:
    axes = np.array([axes]).reshape(3, 2)

# Armazenar resultados e limites globais
stl_results_dV, stl_results_dH = {}, {}
trend_min_v, trend_max_v = np.inf, -np.inf
season_min_v, season_max_v = np.inf, -np.inf
resid_min_v, resid_max_v = np.inf, -np.inf
trend_min_h, trend_max_h = np.inf, -np.inf
season_min_h, season_max_h = np.inf, -np.inf
resid_min_h, resid_max_h = np.inf, -np.inf

for cluster_id in clusters_ordered:
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()

    # --- Séries médias ---
    cluster_data = agg[agg['cell_id'].isin(cluster_cells)]
    pivot_v = cluster_data.pivot(index='cell_id', columns='date', values='dV').fillna(0)
    pivot_h = cluster_data.pivot(index='cell_id', columns='date', values='dH').fillna(0)
    mean_v = pivot_v.mean(axis=0)
    mean_h = pivot_h.mean(axis=0)

    # --- STL dV ---
    ts_v = pd.Series(mean_v.values, index=pd.to_datetime(mean_v.index)).asfreq('MS').interpolate()
    stl_v = STL(ts_v, period=12, robust=True)
    res_v = stl_v.fit()
    stl_results_dV[cluster_id] = res_v

    trend_min_v, trend_max_v = min(trend_min_v, res_v.trend.min()), max(trend_max_v, res_v.trend.max())
    season_min_v, season_max_v = min(season_min_v, res_v.seasonal.min()), max(season_max_v, res_v.seasonal.max())
    resid_min_v, resid_max_v = min(resid_min_v, res_v.resid.min()), max(resid_max_v, res_v.resid.max())

    # --- STL dH ---
    ts_h = pd.Series(mean_h.values, index=pd.to_datetime(mean_h.index)).asfreq('MS').interpolate()
    stl_h = STL(ts_h, period=12, robust=True)
    res_h = stl_h.fit()
    stl_results_dH[cluster_id] = res_h

    trend_min_h, trend_max_h = min(trend_min_h, res_h.trend.min()), max(trend_max_h, res_h.trend.max())
    season_min_h, season_max_h = min(season_min_h, res_h.seasonal.min()), max(season_max_h, res_h.seasonal.max())
    resid_min_h, resid_max_h = min(resid_min_h, res_h.resid.min()), max(resid_max_h, res_h.resid.max())

# Plotar: cada cluster ocupa duas colunas (dV à esquerda, dH à direita)
for c_idx, cluster_id in enumerate(clusters_ordered):
    color = cluster_colors[cluster_id]
    res_v = stl_results_dV[cluster_id]
    res_h = stl_results_dH[cluster_id]

    # --- dV (coluna 2*c_idx) ---
    axes[0, 2*c_idx].plot(res_v.trend, color=color, linewidth=2.5)
    axes[0, 2*c_idx].set_ylim(trend_min_v, trend_max_v)
    axes[0, 2*c_idx].set_title(f"Cluster {cluster_id} – dV (Tendência)", fontsize=12)

    axes[1, 2*c_idx].plot(res_v.seasonal, color=color, linewidth=2.0)
    axes[1, 2*c_idx].set_ylim(season_min_v, season_max_v)
    axes[1, 2*c_idx].set_title(f"Sazonal", fontsize=11)

    axes[2, 2*c_idx].plot(res_v.resid, color=color, linewidth=1.8)
    axes[2, 2*c_idx].set_ylim(resid_min_v, resid_max_v)
    axes[2, 2*c_idx].set_title(f"Resíduo", fontsize=11)

    # --- dH (coluna 2*c_idx + 1) ---
    axes[0, 2*c_idx + 1].plot(res_h.trend, color=color, linewidth=2.5)
    axes[0, 2*c_idx + 1].set_ylim(trend_min_h, trend_max_h)
    axes[0, 2*c_idx + 1].set_title(f"Cluster {cluster_id} – dH (Tendência)", fontsize=12)

    axes[1, 2*c_idx + 1].plot(res_h.seasonal, color=color, linewidth=2.0)
    axes[1, 2*c_idx + 1].set_ylim(season_min_h, season_max_h)
    axes[1, 2*c_idx + 1].set_title(f"Sazonal", fontsize=11)

    axes[2, 2*c_idx + 1].plot(res_h.resid, color=color, linewidth=1.8)
    axes[2, 2*c_idx + 1].set_ylim(resid_min_h, resid_max_h)
    axes[2, 2*c_idx + 1].set_title(f"Resíduo", fontsize=11)

# Etiquetas de linhas
for row, label in enumerate(["Tendência", "Sazonalidade", "Resíduo"]):
    axes[row, 0].set_ylabel(label + " (dV/dH)")

for ax in axes.flatten():
    ax.grid(False)

plt.suptitle("Decomposição STL Comparativa – Deslocamento Vertical (dV) vs Horizontal (dH)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Grelha: {grid_size} m x {grid_size} m',
#     f'Clustering: DTW + Hierárquico (ligação média)',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)

ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais médias por cluster + temperatura ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais do cluster (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário, preta)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

from statsmodels.tsa.seasonal import STL

# ==============================
# 12.1. Mapas e séries médias comparativas (dV vs dH)
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from matplotlib.colors import to_hex
import seaborn as sns
import matplotlib.dates as mdates

# --- 1️⃣ Clustering DTW + Hierárquico para dH ---
agg_pivot_h = agg.pivot(index='cell_id', columns='date', values='dH').fillna(0)
cell_ids_h = agg_pivot_h.index.tolist()
Xh = agg_pivot_h.values.astype(float)
n_h = Xh.shape[0]

# Normalização
from sklearn.preprocessing import StandardScaler
scaler_h = StandardScaler()
Xh_scaled = scaler_h.fit_transform(Xh)

# DTW pairwise (com janela)
dist_matrix_h = np.zeros((n_h, n_h))
window_h = int(0.1 * Xh_scaled.shape[1])
for i in range(n_h):
    xi = Xh_scaled[i, :]
    for j in range(i + 1, n_h):
        xj = Xh_scaled[j, :]
        dist_matrix_h[i, j] = dtw.distance(xi, xj, window=window_h)
        dist_matrix_h[j, i] = dist_matrix_h[i, j]

# Hierárquico
condensed_h = squareform(dist_matrix_h)
Zh = linkage(condensed_h, method='average')

# Corte automático
distances_h = Zh[:, 2]
diffs_h = np.diff(distances_h)
max_jump_idx_h = np.argmax(diffs_h)
cut_distance_h = (distances_h[max_jump_idx_h] + distances_h[max_jump_idx_h - 1]) / 2

cluster_labels_h = fcluster(Zh, t=cut_distance_h, criterion='distance')
num_clusters_h = len(np.unique(cluster_labels_h))
print(f"[dH] Corte ajustado em {cut_distance_h:.3f}, clusters encontrados: {num_clusters_h}")

cluster_df_h = pd.DataFrame({'cell_id': cell_ids_h, 'cluster': cluster_labels_h})
palette_h = sns.color_palette("Set2", num_clusters_h)
cluster_colors_h = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels_h), palette_h)}

grid_sel_h = grid.merge(cluster_df_h, on='cell_id', how='left')

# --- 2️⃣ Mapas comparativos (dV vs dH) ---
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# --- Mapa dV ---
ax1 = axes[0]
grid.boundary.plot(ax=ax1, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax1, color='black', linewidth=0.8, alpha=0.2)
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax1, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )
gdf_points.plot(ax=ax1, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax1, source=ctx.providers.Esri.WorldImagery)
ax1.set_title(f"Clusters de deslocamento vertical (dV)\nDTW + Hierárquico", fontsize=14)
ax1.set_axis_off()

# Legenda dV
handles_v = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=8, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles_v.append(plt.Line2D([0], [0], color=color, linewidth=6, label=f'Cluster {cl}'))
ax1.legend(handles=handles_v, loc='upper left', fontsize=9, framealpha=0.9)

# --- Mapa dH ---
ax2 = axes[1]
grid.boundary.plot(ax=ax2, color='lightgray', linewidth=0.5)
grid_sel_h.boundary.plot(ax=ax2, color='black', linewidth=0.8, alpha=0.2)
for _, row in grid_sel_h.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel_h.crs).plot(
            ax=ax2, color=cluster_colors_h[int(row['cluster'])], alpha=0.4
        )
gdf_points.plot(ax=ax2, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax2, source=ctx.providers.Esri.WorldImagery)
ax2.set_title(f"Clusters de deslocamento horizontal (dH)\nDTW + Hierárquico", fontsize=14)
ax2.set_axis_off()

# Legenda dH
handles_h = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=8, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors_h.items():
    handles_h.append(plt.Line2D([0], [0], color=color, linewidth=6, label=f'Cluster {cl}'))
ax2.legend(handles=handles_h, loc='upper left', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()

# --- 3️⃣ Séries médias (dV e dH lado a lado, com séries individuais a cinza) ---
clusters_present_v = sorted(cluster_df['cluster'].unique())
clusters_present_h = sorted(cluster_df_h['cluster'].unique())
n_clusters_v = len(clusters_present_v)
n_clusters_h = len(clusters_present_h)

fig, axes = plt.subplots(2, max(n_clusters_v, n_clusters_h), figsize=(6 * max(n_clusters_v, n_clusters_h), 10))

# Garantir formato 2D
if n_clusters_v == 1 and n_clusters_h == 1:
    axes = np.array([[axes[0]], [axes[1]]])

# --- Linha 1: dV ---
for idx, cluster_id in enumerate(clusters_present_v):
    ax = axes[0, idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)
    cluster_mean = cluster_data.mean(axis=0)
    color = cluster_colors[cluster_id]
    ax.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')
    ax.set_title(f"Cluster {cluster_id} – dV\n({len(cluster_cells)} células)", fontsize=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

# --- Linha 2: dH ---
for idx, cluster_id in enumerate(clusters_present_h):
    ax = axes[1, idx]
    cluster_cells = cluster_df_h[cluster_df_h['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_h.loc[cluster_cells]
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)
    cluster_mean = cluster_data.mean(axis=0)
    color = cluster_colors_h[cluster_id]
    ax.plot(cluster_mean.index, cluster_mean.values, color=color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')
    ax.set_title(f"Cluster {cluster_id} – dH\n({len(cluster_cells)} células)", fontsize=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dH (mm)")
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.suptitle("Séries temporais médias por cluster – dV (topo) e dH (base)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


# ==============================
# 12.2. Dendrogramas e correlação com temperatura (dV e dH)
# ==============================
import scipy.stats as stats
from scipy.cluster.hierarchy import dendrogram

# --- 1️⃣ Dendrogramas DTW (dV e dH) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Dendrograma dV
dendrogram(Z, labels=cell_ids, color_threshold=cut_distance, ax=axes[0])
axes[0].set_title("Dendrograma – deslocamento vertical (dV)", fontsize=13)
axes[0].set_xlabel("Células")
axes[0].set_ylabel("Distância DTW")
axes[0].axhline(y=cut_distance, color='red', linestyle='--', label='Corte automático')
axes[0].legend(fontsize=9)

# Dendrograma dH
dendrogram(Zh, labels=cell_ids_h, color_threshold=cut_distance_h, ax=axes[1])
axes[1].set_title("Dendrograma – deslocamento horizontal (dH)", fontsize=13)
axes[1].set_xlabel("Células")
axes[1].set_ylabel("Distância DTW")
axes[1].axhline(y=cut_distance_h, color='red', linestyle='--', label='Corte automático')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# --- 2️⃣ Correlação com temperatura (por cluster, dV e dH) ---

# Temperatura agregada por data (assumindo que existe uma coluna 'temperature' no DataFrame original)
# Se não existir, ajusta aqui conforme o nome real da tua variável de temperatura
if 'temperature' in agg.columns:
    temp_series = agg.groupby('date')['temperature'].mean()
else:
    print("⚠️ Atenção: coluna 'temperature' não encontrada — correlação não será calculada.")
    temp_series = None

if temp_series is not None:
    # --- Correlação para dV ---
    corr_results_v = []
    for cluster_id in sorted(cluster_df['cluster'].unique()):
        cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
        cluster_data = agg_pivot.loc[cluster_cells].mean(axis=0)
        corr, pval = stats.pearsonr(cluster_data.values, temp_series.loc[cluster_data.index].values)
        corr_results_v.append((cluster_id, corr, pval))

    # --- Correlação para dH ---
    corr_results_h = []
    for cluster_id in sorted(cluster_df_h['cluster'].unique()):
        cluster_cells = cluster_df_h[cluster_df_h['cluster'] == cluster_id]['cell_id'].tolist()
        cluster_data = agg_pivot_h.loc[cluster_cells].mean(axis=0)
        corr, pval = stats.pearsonr(cluster_data.values, temp_series.loc[cluster_data.index].values)
        corr_results_h.append((cluster_id, corr, pval))

    # Tabelas resumo
    corr_df_v = pd.DataFrame(corr_results_v, columns=['Cluster', 'r (dV)', 'p-value'])
    corr_df_h = pd.DataFrame(corr_results_h, columns=['Cluster', 'r (dH)', 'p-value'])
    corr_summary = corr_df_v.merge(corr_df_h, on='Cluster', how='outer')

    print("=== Correlação com temperatura média ===")
    display(corr_summary)

    # --- 3️⃣ Visualização das correlações ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # dV
    axes[0].bar(corr_df_v['Cluster'], corr_df_v['r (dV)'], color=[cluster_colors[c] for c in corr_df_v['Cluster']])
    axes[0].axhline(0, color='black', linewidth=0.8)
    axes[0].set_title("Correlação dV vs Temperatura", fontsize=13)
    axes[0].set_ylabel("Coeficiente de correlação (r)")
    axes[0].set_xlabel("Cluster")
    axes[0].grid(alpha=0.3)

    # dH
    axes[1].bar(corr_df_h['Cluster'], corr_df_h['r (dH)'], color=[cluster_colors_h[c] for c in corr_df_h['Cluster']])
    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set_title("Correlação dH vs Temperatura", fontsize=13)
    axes[1].set_ylabel("Coeficiente de correlação (r)")
    axes[1].set_xlabel("Cluster")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# ==============================
# 13. Decomposição STL comparativa (dV vs dH)
# ==============================

# Clusters ordenados: cluster 1 primeiro
clusters_ordered = sorted(clusters_present)
if 1 in clusters_ordered:
    clusters_ordered.remove(1)
    clusters_ordered = [1] + clusters_ordered

# Configuração da figura: 3 linhas (tendência/sazonal/resíduo), 2*n_clusters colunas (dV e dH lado a lado)
fig_stl, axes = plt.subplots(3, 2 * n_clusters, figsize=(5 * 2 * n_clusters, 10), sharex=True)

if n_clusters == 1:
    axes = np.array([axes]).reshape(3, 2)

# Armazenar resultados e limites globais
stl_results_dV, stl_results_dH = {}, {}
trend_min_v, trend_max_v = np.inf, -np.inf
season_min_v, season_max_v = np.inf, -np.inf
resid_min_v, resid_max_v = np.inf, -np.inf
trend_min_h, trend_max_h = np.inf, -np.inf
season_min_h, season_max_h = np.inf, -np.inf
resid_min_h, resid_max_h = np.inf, -np.inf

for cluster_id in clusters_ordered:
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()

    # --- Séries médias ---
    cluster_data = agg[agg['cell_id'].isin(cluster_cells)]
    pivot_v = cluster_data.pivot(index='cell_id', columns='date', values='dV').fillna(0)
    pivot_h = cluster_data.pivot(index='cell_id', columns='date', values='dH').fillna(0)
    mean_v = pivot_v.mean(axis=0)
    mean_h = pivot_h.mean(axis=0)

    # --- STL dV ---
    ts_v = pd.Series(mean_v.values, index=pd.to_datetime(mean_v.index)).asfreq('MS').interpolate()
    stl_v = STL(ts_v, period=12, robust=True)
    res_v = stl_v.fit()
    stl_results_dV[cluster_id] = res_v

    trend_min_v, trend_max_v = min(trend_min_v, res_v.trend.min()), max(trend_max_v, res_v.trend.max())
    season_min_v, season_max_v = min(season_min_v, res_v.seasonal.min()), max(season_max_v, res_v.seasonal.max())
    resid_min_v, resid_max_v = min(resid_min_v, res_v.resid.min()), max(resid_max_v, res_v.resid.max())

    # --- STL dH ---
    ts_h = pd.Series(mean_h.values, index=pd.to_datetime(mean_h.index)).asfreq('MS').interpolate()
    stl_h = STL(ts_h, period=12, robust=True)
    res_h = stl_h.fit()
    stl_results_dH[cluster_id] = res_h

    trend_min_h, trend_max_h = min(trend_min_h, res_h.trend.min()), max(trend_max_h, res_h.trend.max())
    season_min_h, season_max_h = min(season_min_h, res_h.seasonal.min()), max(season_max_h, res_h.seasonal.max())
    resid_min_h, resid_max_h = min(resid_min_h, res_h.resid.min()), max(resid_max_h, res_h.resid.max())

# Plotar: cada cluster ocupa duas colunas (dV à esquerda, dH à direita)
for c_idx, cluster_id in enumerate(clusters_ordered):
    color = cluster_colors[cluster_id]
    res_v = stl_results_dV[cluster_id]
    res_h = stl_results_dH[cluster_id]

    # --- dV (coluna 2*c_idx) ---
    axes[0, 2*c_idx].plot(res_v.trend, color=color, linewidth=2.5)
    axes[0, 2*c_idx].set_ylim(trend_min_v, trend_max_v)
    axes[0, 2*c_idx].set_title(f"Cluster {cluster_id} – dV (Tendência)", fontsize=12)

    axes[1, 2*c_idx].plot(res_v.seasonal, color=color, linewidth=2.0)
    axes[1, 2*c_idx].set_ylim(season_min_v, season_max_v)
    axes[1, 2*c_idx].set_title(f"Sazonal", fontsize=11)

    axes[2, 2*c_idx].plot(res_v.resid, color=color, linewidth=1.8)
    axes[2, 2*c_idx].set_ylim(resid_min_v, resid_max_v)
    axes[2, 2*c_idx].set_title(f"Resíduo", fontsize=11)

    # --- dH (coluna 2*c_idx + 1) ---
    axes[0, 2*c_idx + 1].plot(res_h.trend, color=color, linewidth=2.5)
    axes[0, 2*c_idx + 1].set_ylim(trend_min_h, trend_max_h)
    axes[0, 2*c_idx + 1].set_title(f"Cluster {cluster_id} – dH (Tendência)", fontsize=12)

    axes[1, 2*c_idx + 1].plot(res_h.seasonal, color=color, linewidth=2.0)
    axes[1, 2*c_idx + 1].set_ylim(season_min_h, season_max_h)
    axes[1, 2*c_idx + 1].set_title(f"Sazonal", fontsize=11)

    axes[2, 2*c_idx + 1].plot(res_h.resid, color=color, linewidth=1.8)
    axes[2, 2*c_idx + 1].set_ylim(resid_min_h, resid_max_h)
    axes[2, 2*c_idx + 1].set_title(f"Resíduo", fontsize=11)

# Etiquetas de linhas
for row, label in enumerate(["Tendência", "Sazonalidade", "Resíduo"]):
    axes[row, 0].set_ylabel(label + " (dV/dH)")

for ax in axes.flatten():
    ax.grid(False)

plt.suptitle("Decomposição STL Comparativa – Deslocamento Vertical (dV) vs Horizontal (dH)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ================================================
# 12. Análise de Clusters – versão final e corrigida
# ================================================
import scipy.stats as stats
from scipy.cluster.hierarchy import dendrogram
import matplotlib.pyplot as plt
import contextily as ctx
import numpy as np
import pandas as pd

# =======================================================
# 12.1 Dendrogramas DTW (dV e dH)
# =======================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# dV
dendrogram(Z, labels=cell_ids, color_threshold=cut_distance, ax=axes[0])
axes[0].set_title("Dendrograma – deslocamento vertical (dV)", fontsize=13)
axes[0].set_xlabel("Células")
axes[0].set_ylabel("Distância DTW")
axes[0].axhline(y=cut_distance, color='red', linestyle='--', label='Corte automático')
axes[0].legend(fontsize=9)

# dH
dendrogram(Zh, labels=cell_ids_h, color_threshold=cut_distance_h, ax=axes[1])
axes[1].set_title("Dendrograma – deslocamento horizontal (dH)", fontsize=13)
axes[1].set_xlabel("Células")
axes[1].set_ylabel("Distância DTW")
axes[1].axhline(y=cut_distance_h, color='red', linestyle='--', label='Corte automático')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()


# =======================================================
# 12.2 Mapas de clusters (dV e dH)
# =======================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# --- dV ---
grid.merge(cluster_df, on='cell_id').plot(
    column='cluster',
    categorical=True,
    cmap='tab10',
    legend=True,
    ax=axes[0],
    edgecolor='none'
)
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].set_title("Clusters de deslocamento vertical (dV)", fontsize=13)
axes[0].axis('off')

# --- dH ---
grid.merge(cluster_df_h, on='cell_id').plot(
    column='cluster',
    categorical=True,
    cmap='tab10',
    legend=True,
    ax=axes[1],
    edgecolor='none'
)
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].set_title("Clusters de deslocamento horizontal (dH)", fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.show()


# =======================================================
# 12.3 Séries médias por cluster (dV e dH)
# =======================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- dV ---
for cluster_id in sorted(cluster_df['cluster'].unique()):
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id']
    for c in cluster_cells:
        axes[0].plot(agg_pivot.columns, agg_pivot.loc[c], color='lightgray', alpha=0.5)
    mean_series = agg_pivot.loc[cluster_cells].mean(axis=0)
    axes[0].plot(agg_pivot.columns, mean_series, label=f'Cluster {cluster_id}', linewidth=2)
axes[0].set_title("Deslocamento vertical (dV)", fontsize=13)
axes[0].set_xlabel("Data")
axes[0].set_ylabel("Deslocamento (mm)")
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# --- dH ---
for cluster_id in sorted(cluster_df_h['cluster'].unique()):
    cluster_cells = cluster_df_h[cluster_df_h['cluster'] == cluster_id]['cell_id']
    for c in cluster_cells:
        axes[1].plot(agg_pivot_h.columns, agg_pivot_h.loc[c], color='lightgray', alpha=0.5)
    mean_series = agg_pivot_h.loc[cluster_cells].mean(axis=0)
    axes[1].plot(agg_pivot_h.columns, mean_series, label=f'Cluster {cluster_id}', linewidth=2)
axes[1].set_title("Deslocamento horizontal (dH)", fontsize=13)
axes[1].set_xlabel("Data")
axes[1].set_ylabel("Deslocamento (mm)")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


# =======================================================
# 12.4 Correlação com temperatura (df_temp['med_smooth'])
# =======================================================
if 'med_smooth' in df_temp.columns:
    temp_series = df_temp['med_smooth']
    temp_series.index = pd.to_datetime(df_temp['date'])
else:
    raise ValueError("⚠️ A coluna 'med_smooth' não foi encontrada em df_temp!")

# --- dV ---
corr_results_v = []
for cluster_id in sorted(cluster_df['cluster'].unique()):
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_mean = agg_pivot.loc[cluster_cells].mean(axis=0)
    cluster_mean.index = pd.to_datetime(cluster_mean.index)
    corr, pval = stats.pearsonr(cluster_mean.loc[temp_series.index].values, temp_series.values)
    corr_results_v.append((cluster_id, corr, pval))

# --- dH ---
corr_results_h = []
for cluster_id in sorted(cluster_df_h['cluster'].unique()):
    cluster_cells = cluster_df_h[cluster_df_h['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_mean = agg_pivot_h.loc[cluster_cells].mean(axis=0)
    cluster_mean.index = pd.to_datetime(cluster_mean.index)
    corr, pval = stats.pearsonr(cluster_mean.loc[temp_series.index].values, temp_series.values)
    corr_results_h.append((cluster_id, corr, pval))

corr_df_v = pd.DataFrame(corr_results_v, columns=['Cluster', 'r (dV)', 'p-value'])
corr_df_h = pd.DataFrame(corr_results_h, columns=['Cluster', 'r (dH)', 'p-value'])
corr_summary = corr_df_v.merge(corr_df_h, on='Cluster', how='outer')

print("=== Correlação com temperatura (med_smooth) ===")
display(corr_summary)

# --- Gráfico das correlações ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(corr_df_v['Cluster'], corr_df_v['r (dV)'], color='tab:blue')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title("Correlação dV vs Temperatura", fontsize=13)
axes[0].set_ylabel("Coeficiente de correlação (r)")
axes[0].set_xlabel("Cluster")
axes[0].grid(alpha=0.3)

axes[1].bar(corr_df_h['Cluster'], corr_df_h['r (dH)'], color='tab:orange')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title("Correlação dH vs Temperatura", fontsize=13)
axes[1].set_ylabel("Coeficiente de correlação (r)")
axes[1].set_xlabel("Cluster")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


# =======================================================
# 12.5 Decomposição STL comparativa (ajustada)
# =======================================================
from statsmodels.tsa.seasonal import STL

def plot_stl_comparison(cluster_df, agg_pivot, title_prefix, ax):
    for cluster_id in sorted(cluster_df['cluster'].unique()):
        cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id']
        mean_series = agg_pivot.loc[cluster_cells].mean(axis=0)
        mean_series.index = pd.to_datetime(mean_series.index)
        stl = STL(mean_series, period=12).fit()
        ax.plot(stl.trend, label=f'Tendência Cluster {cluster_id}', linewidth=2)
    ax.set_title(f'{title_prefix} – Tendência STL', fontsize=13)
    ax.set_xlabel("Data")
    ax.set_ylabel("Deslocamento (mm)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_stl_comparison(cluster_df, agg_pivot, "dV", axes[0])
plot_stl_comparison(cluster_df_h, agg_pivot_h, "dH", axes[1])
plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries médias (dV e dH lado a lado) vs temperatura
# ==============================

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(3, 2 * n_clusters, height_ratios=[2, 1, 1])

# --- Linha 1: Mapa com clusters ---
ax_map = fig.add_subplot(gs[0, :])

grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Legenda
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    "Clusters de séries temporais de deslocamento vertical (dV) e horizontal (dH)\n"
    "Comparação com temperatura média suavizada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# ==============================
# Linhas 2 e 3: dV e dH por cluster
# ==============================

# Pivot para dV e dH
agg_pivot_v = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
agg_pivot_h = agg.pivot(index='cell_id', columns='date', values='dH').fillna(0)

disp_min = min(agg['dV'].min(), agg['dH'].min())
disp_max = max(agg['dV'].max(), agg['dH'].max())
disp_margin = (disp_max - disp_min) * 0.1 if (disp_max - disp_min) > 0 else 1.0

temp_min, temp_max = df_temp['med_smooth'].min(), df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_color = cluster_colors[cluster_id]

    # ================== dV ==================
    ax_v = fig.add_subplot(gs[1, 2*idx])
    cluster_data_v = agg_pivot_v.loc[cluster_cells]
    for cid in cluster_data_v.index:
        ax_v.plot(cluster_data_v.columns, cluster_data_v.loc[cid], color='lightgray', alpha=0.5)
    cluster_mean_v = cluster_data_v.mean(axis=0)
    ax_v.plot(cluster_mean_v.index, cluster_mean_v.values, color=cluster_color, linewidth=2.5, label='Média dV')

    # Temperatura (eixo secundário)
    ax_v2 = ax_v.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (disp_max - disp_min) * 0.25 + (disp_max - (disp_max - disp_min) * 0.3)
    ax_v2.plot(df_temp['data'], temp_visual, color='gray', linestyle='--', linewidth=1.8, label='Temperatura (°C)')

    ax_v.set_ylim(disp_min - disp_margin, disp_max + disp_margin)
    ax_v2.set_ylim(disp_min - disp_margin, disp_max + disp_margin)
    ax_v.set_title(f"Cluster {cluster_id} — dV", fontsize=12)
    ax_v.set_ylabel("dV (mm)")
    ax_v.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax_v.grid(False)
    ax_v2.grid(False)

    # Combinar legendas
    lines1, labels1 = ax_v.get_legend_handles_labels()
    lines2, labels2 = ax_v2.get_legend_handles_labels()
    ax_v.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

    # ================== dH ==================
    ax_h = fig.add_subplot(gs[1, 2*idx + 1])
    cluster_data_h = agg_pivot_h.loc[cluster_cells]
    for cid in cluster_data_h.index:
        ax_h.plot(cluster_data_h.columns, cluster_data_h.loc[cid], color='lightgray', alpha=0.5)
    cluster_mean_h = cluster_data_h.mean(axis=0)
    ax_h.plot(cluster_mean_h.index, cluster_mean_h.values, color=cluster_color, linewidth=2.5, label='Média dH')

    # Temperatura (eixo secundário)
    ax_h2 = ax_h.twinx()
    ax_h2.plot(df_temp['data'], temp_visual, color='gray', linestyle='--', linewidth=1.8, label='Temperatura (°C)')

    ax_h.set_ylim(disp_min - disp_margin, disp_max + disp_margin)
    ax_h2.set_ylim(disp_min - disp_margin, disp_max + disp_margin)
    ax_h.set_title(f"Cluster {cluster_id} — dH", fontsize=12)
    ax_h.set_ylabel("dH (mm)")
    ax_h.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax_h.grid(False)
    ax_h2.grid(False)

    # Combinar legendas
    lines1, labels1 = ax_h.get_legend_handles_labels()
    lines2, labels2 = ax_h2.get_legend_handles_labels()
    ax_h.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()



from statsmodels.tsa.seasonal import STL

# ==============================
# 13. Decomposição STL comparativa (dV vs dH)
# ==============================

# Clusters ordenados: cluster 1 primeiro
clusters_ordered = sorted(clusters_present)
if 1 in clusters_ordered:
    clusters_ordered.remove(1)
    clusters_ordered = [1] + clusters_ordered

# Configuração da figura: 3 linhas (tendência/sazonal/resíduo), 2*n_clusters colunas (dV e dH lado a lado)
fig_stl, axes = plt.subplots(3, 2 * n_clusters, figsize=(5 * 2 * n_clusters, 10), sharex=True)

if n_clusters == 1:
    axes = np.array([axes]).reshape(3, 2)

# Armazenar resultados e limites globais
stl_results_dV, stl_results_dH = {}, {}
trend_min_v, trend_max_v = np.inf, -np.inf
season_min_v, season_max_v = np.inf, -np.inf
resid_min_v, resid_max_v = np.inf, -np.inf
trend_min_h, trend_max_h = np.inf, -np.inf
season_min_h, season_max_h = np.inf, -np.inf
resid_min_h, resid_max_h = np.inf, -np.inf

for cluster_id in clusters_ordered:
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()

    # --- Séries médias ---
    cluster_data = agg[agg['cell_id'].isin(cluster_cells)]
    pivot_v = cluster_data.pivot(index='cell_id', columns='date', values='dV').fillna(0)
    pivot_h = cluster_data.pivot(index='cell_id', columns='date', values='dH').fillna(0)
    mean_v = pivot_v.mean(axis=0)
    mean_h = pivot_h.mean(axis=0)

    # --- STL dV ---
    ts_v = pd.Series(mean_v.values, index=pd.to_datetime(mean_v.index)).asfreq('MS').interpolate()
    stl_v = STL(ts_v, period=12, robust=True)
    res_v = stl_v.fit()
    stl_results_dV[cluster_id] = res_v

    trend_min_v, trend_max_v = min(trend_min_v, res_v.trend.min()), max(trend_max_v, res_v.trend.max())
    season_min_v, season_max_v = min(season_min_v, res_v.seasonal.min()), max(season_max_v, res_v.seasonal.max())
    resid_min_v, resid_max_v = min(resid_min_v, res_v.resid.min()), max(resid_max_v, res_v.resid.max())

    # --- STL dH ---
    ts_h = pd.Series(mean_h.values, index=pd.to_datetime(mean_h.index)).asfreq('MS').interpolate()
    stl_h = STL(ts_h, period=12, robust=True)
    res_h = stl_h.fit()
    stl_results_dH[cluster_id] = res_h

    trend_min_h, trend_max_h = min(trend_min_h, res_h.trend.min()), max(trend_max_h, res_h.trend.max())
    season_min_h, season_max_h = min(season_min_h, res_h.seasonal.min()), max(season_max_h, res_h.seasonal.max())
    resid_min_h, resid_max_h = min(resid_min_h, res_h.resid.min()), max(resid_max_h, res_h.resid.max())

# Plotar: cada cluster ocupa duas colunas (dV à esquerda, dH à direita)
for c_idx, cluster_id in enumerate(clusters_ordered):
    color = cluster_colors[cluster_id]
    res_v = stl_results_dV[cluster_id]
    res_h = stl_results_dH[cluster_id]

    # --- dV (coluna 2*c_idx) ---
    axes[0, 2*c_idx].plot(res_v.trend, color=color, linewidth=2.5)
    axes[0, 2*c_idx].set_ylim(trend_min_v, trend_max_v)
    axes[0, 2*c_idx].set_title(f"Cluster {cluster_id} – dV (Tendência)", fontsize=12)

    axes[1, 2*c_idx].plot(res_v.seasonal, color=color, linewidth=2.0)
    axes[1, 2*c_idx].set_ylim(season_min_v, season_max_v)
    axes[1, 2*c_idx].set_title(f"Sazonal", fontsize=11)

    axes[2, 2*c_idx].plot(res_v.resid, color=color, linewidth=1.8)
    axes[2, 2*c_idx].set_ylim(resid_min_v, resid_max_v)
    axes[2, 2*c_idx].set_title(f"Resíduo", fontsize=11)

    # --- dH (coluna 2*c_idx + 1) ---
    axes[0, 2*c_idx + 1].plot(res_h.trend, color=color, linewidth=2.5)
    axes[0, 2*c_idx + 1].set_ylim(trend_min_h, trend_max_h)
    axes[0, 2*c_idx + 1].set_title(f"Cluster {cluster_id} – dH (Tendência)", fontsize=12)

    axes[1, 2*c_idx + 1].plot(res_h.seasonal, color=color, linewidth=2.0)
    axes[1, 2*c_idx + 1].set_ylim(season_min_h, season_max_h)
    axes[1, 2*c_idx + 1].set_title(f"Sazonal", fontsize=11)

    axes[2, 2*c_idx + 1].plot(res_h.resid, color=color, linewidth=1.8)
    axes[2, 2*c_idx + 1].set_ylim(resid_min_h, resid_max_h)
    axes[2, 2*c_idx + 1].set_title(f"Resíduo", fontsize=11)

# Etiquetas de linhas
for row, label in enumerate(["Tendência", "Sazonalidade", "Resíduo"]):
    axes[row, 0].set_ylabel(label + " (dV/dH)")

for ax in axes.flatten():
    ax.grid(False)

plt.suptitle("Decomposição STL Comparativa – Deslocamento Vertical (dV) vs Horizontal (dH)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries médias (dV e dH) vs temperatura (dH incluído)
# ==============================

clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1])  # mapa em cima, séries abaixo (uma linha com dV+dH)

# --- Linha 1: Mapa com clusters ---
ax_map = fig.add_subplot(gs[0, :])

grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Legenda
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais — dV e dH médias por cluster vs Temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# ==============================
# Linha 2: Séries médias (dV e dH) por cluster + temperatura
# ==============================

# Pivot para dV e dH
agg_pivot_v = agg.pivot(index='cell_id', columns='date', values='dV')
agg_pivot_h = agg.pivot(index='cell_id', columns='date', values='dH')

# Substituir NaNs por interpolação por célula (mais seguro do que fillna(0))
agg_pivot_v = agg_pivot_v.apply(lambda col: col.interpolate(axis=0).fillna(method='bfill').fillna(method='ffill'), axis=0)
agg_pivot_h = agg_pivot_h.apply(lambda col: col.interpolate(axis=0).fillna(method='bfill').fillna(method='ffill'), axis=0)

# Escalas comuns (usar ambas dV e dH para um mesmo intervalo visual)
disp_min = min(agg['dV'].min(), agg['dH'].min())
disp_max = max(agg['dV'].max(), agg['dH'].max())
disp_margin = (disp_max - disp_min) * 0.12 if (disp_max - disp_min) > 0 else 1.0  # margem, evita zero-range

temp_min, temp_max = df_temp['med_smooth'].min(), df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_color = cluster_colors[cluster_id]

    # Séries individuais (dV e dH) em cinza muito claro
    # Para dV
    cluster_data_v = agg_pivot_v.reindex(cluster_cells).dropna(how='all')
    if not cluster_data_v.empty:
        for cid in cluster_data_v.index:
            ax.plot(cluster_data_v.columns, cluster_data_v.loc[cid], color='lightgray', alpha=0.4, linewidth=0.8)

    # Para dH
    cluster_data_h = agg_pivot_h.reindex(cluster_cells).dropna(how='all')
    if not cluster_data_h.empty:
        for cid in cluster_data_h.index:
            ax.plot(cluster_data_h.columns, cluster_data_h.loc[cid], color='lightgray', alpha=0.25, linewidth=0.8)

    # Médias do cluster
    cluster_mean_v = cluster_data_v.mean(axis=0) if not cluster_data_v.empty else pd.Series(index=agg_pivot_v.columns, dtype=float)
    cluster_mean_h = cluster_data_h.mean(axis=0) if not cluster_data_h.empty else pd.Series(index=agg_pivot_h.columns, dtype=float)

    # Plot das médias: dV sólido, dH tracejado (mesma cor)
    if cluster_mean_v.notna().any():
        ax.plot(cluster_mean_v.index, cluster_mean_v.values, color=cluster_color, linewidth=2.5, label='Média dV')
    if cluster_mean_h.notna().any():
        ax.plot(cluster_mean_h.index, cluster_mean_h.values, color=cluster_color, linewidth=2.2, linestyle='--', label='Média dH')

    # Temperatura (eixo secundário, cinza)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (disp_max - disp_min) * 0.25 + (disp_max - (disp_max - disp_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='gray', linestyle='--', alpha=0.9, linewidth=1.8, label='Temperatura (°C)')

    # Limites e labels
    ax.set_ylim(disp_min - disp_margin, disp_max + disp_margin)
    ax2.set_ylim(disp_min - disp_margin, disp_max + disp_margin)

    ax.set_title(f'Cluster {cluster_id} — {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel("Deslocamento (mm)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda: combinar linhas da esquerda e direita
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    # garantir legenda logicamente ordenada: Média dV, Média dH, Temperatura
    combined_lines = []
    combined_labels = []
    for lab in ['Média dV', 'Média dH', 'Temperatura (°C)']:
        if lab in labels1 + labels2:
            if lab in labels1:
                i = labels1.index(lab); combined_lines.append(lines1[i]); combined_labels.append(lab)
            else:
                i = labels2.index(lab); combined_lines.append(lines2[i]); combined_labels.append(lab)
    ax.legend(combined_lines, combined_labels, fontsize=9, loc='upper left')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


from statsmodels.tsa.seasonal import STL

# ==============================
# 13. Decomposição STL comparativa (dV vs dH)
# ==============================

# Clusters ordenados: cluster 1 primeiro
clusters_ordered = sorted(clusters_present)
if 1 in clusters_ordered:
    clusters_ordered.remove(1)
    clusters_ordered = [1] + clusters_ordered

# Configuração da figura: 3 linhas (tendência/sazonal/resíduo), 2*n_clusters colunas (dV e dH lado a lado)
fig_stl, axes = plt.subplots(3, 2 * n_clusters, figsize=(5 * 2 * n_clusters, 10), sharex=True)

if n_clusters == 1:
    axes = np.array([axes]).reshape(3, 2)

# Armazenar resultados e limites globais
stl_results_dV, stl_results_dH = {}, {}
trend_min_v, trend_max_v = np.inf, -np.inf
season_min_v, season_max_v = np.inf, -np.inf
resid_min_v, resid_max_v = np.inf, -np.inf
trend_min_h, trend_max_h = np.inf, -np.inf
season_min_h, season_max_h = np.inf, -np.inf
resid_min_h, resid_max_h = np.inf, -np.inf

for cluster_id in clusters_ordered:
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()

    # --- Séries médias ---
    cluster_data = agg[agg['cell_id'].isin(cluster_cells)]
    pivot_v = cluster_data.pivot(index='cell_id', columns='date', values='dV').fillna(0)
    pivot_h = cluster_data.pivot(index='cell_id', columns='date', values='dH').fillna(0)
    mean_v = pivot_v.mean(axis=0)
    mean_h = pivot_h.mean(axis=0)

    # --- STL dV ---
    ts_v = pd.Series(mean_v.values, index=pd.to_datetime(mean_v.index)).asfreq('MS').interpolate()
    stl_v = STL(ts_v, period=12, robust=True)
    res_v = stl_v.fit()
    stl_results_dV[cluster_id] = res_v

    trend_min_v, trend_max_v = min(trend_min_v, res_v.trend.min()), max(trend_max_v, res_v.trend.max())
    season_min_v, season_max_v = min(season_min_v, res_v.seasonal.min()), max(season_max_v, res_v.seasonal.max())
    resid_min_v, resid_max_v = min(resid_min_v, res_v.resid.min()), max(resid_max_v, res_v.resid.max())

    # --- STL dH ---
    ts_h = pd.Series(mean_h.values, index=pd.to_datetime(mean_h.index)).asfreq('MS').interpolate()
    stl_h = STL(ts_h, period=12, robust=True)
    res_h = stl_h.fit()
    stl_results_dH[cluster_id] = res_h

    trend_min_h, trend_max_h = min(trend_min_h, res_h.trend.min()), max(trend_max_h, res_h.trend.max())
    season_min_h, season_max_h = min(season_min_h, res_h.seasonal.min()), max(season_max_h, res_h.seasonal.max())
    resid_min_h, resid_max_h = min(resid_min_h, res_h.resid.min()), max(resid_max_h, res_h.resid.max())

# Plotar: cada cluster ocupa duas colunas (dV à esquerda, dH à direita)
for c_idx, cluster_id in enumerate(clusters_ordered):
    color = cluster_colors[cluster_id]
    res_v = stl_results_dV[cluster_id]
    res_h = stl_results_dH[cluster_id]

    # --- dV (coluna 2*c_idx) ---
    axes[0, 2*c_idx].plot(res_v.trend, color=color, linewidth=2.5)
    axes[0, 2*c_idx].set_ylim(trend_min_v, trend_max_v)
    axes[0, 2*c_idx].set_title(f"Cluster {cluster_id} – dV (Tendência)", fontsize=12)

    axes[1, 2*c_idx].plot(res_v.seasonal, color=color, linewidth=2.0)
    axes[1, 2*c_idx].set_ylim(season_min_v, season_max_v)
    axes[1, 2*c_idx].set_title(f"Sazonal", fontsize=11)

    axes[2, 2*c_idx].plot(res_v.resid, color=color, linewidth=1.8)
    axes[2, 2*c_idx].set_ylim(resid_min_v, resid_max_v)
    axes[2, 2*c_idx].set_title(f"Resíduo", fontsize=11)

    # --- dH (coluna 2*c_idx + 1) ---
    axes[0, 2*c_idx + 1].plot(res_h.trend, color=color, linewidth=2.5)
    axes[0, 2*c_idx + 1].set_ylim(trend_min_h, trend_max_h)
    axes[0, 2*c_idx + 1].set_title(f"Cluster {cluster_id} – dH (Tendência)", fontsize=12)

    axes[1, 2*c_idx + 1].plot(res_h.seasonal, color=color, linewidth=2.0)
    axes[1, 2*c_idx + 1].set_ylim(season_min_h, season_max_h)
    axes[1, 2*c_idx + 1].set_title(f"Sazonal", fontsize=11)

    axes[2, 2*c_idx + 1].plot(res_h.resid, color=color, linewidth=1.8)
    axes[2, 2*c_idx + 1].set_ylim(resid_min_h, resid_max_h)
    axes[2, 2*c_idx + 1].set_title(f"Resíduo", fontsize=11)

# Etiquetas de linhas
for row, label in enumerate(["Tendência", "Sazonalidade", "Resíduo"]):
    axes[row, 0].set_ylabel(label + " (dV/dH)")

for ax in axes.flatten():
    ax.grid(False)

plt.suptitle("Decomposição STL Comparativa – Deslocamento Vertical (dV) vs Horizontal (dH)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Grelha: {grid_size} m x {grid_size} m',
#     f'Clustering: DTW + Hierárquico (ligação média)',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)

ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais médias por cluster + temperatura ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais do cluster (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário, preta)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais se ainda não existir
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))
clusters_present = sorted(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """Determina a cor de cada ramo do dendrograma conforme os clusters."""
    if max(y_coords) > cut_distance:
        return "#A0A0A0"  # acima do corte
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]
    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    return "#A0A0A0"

# ---------------------------------------------------
# FIGURA 1: Dendrograma + Mapa com legenda e descrição técnica
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.02)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)
leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round', zorder=3)

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW', zorder=2)
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")

# Colorir labels das folhas conforme o cluster
ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors.get(cluster_id, "#333333"))

ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# 👉 Adiciona esta linha para incluir na legenda
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id} - {len(cluster_df[cluster_df["cluster"]==cluster_id])} células')

# # --- Caixa técnica
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico (método "average")',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)

ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# FIGURA 2: Séries médias por cluster + temperatura
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(clusters_present):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cluster_cells) == 0:
        ax.set_title(f'Cluster {cluster_id} - 0 células')
        continue

    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.8,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()



### Hierárquico (ligação completa) + DTW, k=auto=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='complete')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Grelha: {grid_size} m x {grid_size} m',
#     f'Clustering: DTW + Hierárquico (ligação média)',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação completa.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais médias por cluster + temperatura ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais do cluster (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário, preta)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais se ainda não existir
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))
clusters_present = sorted(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """Determina a cor de cada ramo do dendrograma conforme os clusters."""
    if max(y_coords) > cut_distance:
        return "#A0A0A0"  # acima do corte
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]
    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    return "#A0A0A0"

# ---------------------------------------------------
# FIGURA 1: Dendrograma + Mapa com legenda e descrição técnica
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.02)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)
leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round', zorder=3)

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW', zorder=2)
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")

# Colorir labels das folhas conforme o cluster
ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors.get(cluster_id, "#333333"))

ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# 👉 Adiciona esta linha para incluir na legenda
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id} - {len(cluster_df[cluster_df["cluster"]==cluster_id])} células')

# # --- Caixa técnica
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico (método "average")',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação completa.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# FIGURA 2: Séries médias por cluster + temperatura
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(clusters_present):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cluster_cells) == 0:
        ax.set_title(f'Cluster {cluster_id} - 0 células')
        continue

    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.8,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K-Means, k=2, grid = 100 m, dV

### K-Means, k=2, grid = 100 m, dH

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering de dH
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dH').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters (dH)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento horizontal.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ------------------------------
# Linha 2: Séries temporais
# ------------------------------
# Escala comum para dH
dH_min = agg['dH'].min()
dH_max = agg['dH'].max()
dH_margin = (dH_max - dH_min) * 0.1  # margem de 10%

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # Selecionar células do cluster
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Plot das células individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (linha colorida)
    cluster_mean_dH = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dH, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Temperatura média (suavizada e reescalada) ----------
    ax2 = ax.twinx()

    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()

    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dH_max - dH_min) * 0.25 + (dH_max - (dH_max - dH_min) * 0.3)

    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85, label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Definir ticks reais
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dH_max - dH_min) * 0.25 + (dH_max - (dH_max - dH_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])

    # ---------- Eixos e estilo ----------
    ax.set_ylim(dH_min - dH_margin, dH_max + dH_margin)
    ax2.set_ylim(dH_min - dH_margin, dH_max + dH_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dH (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # ---------- Legenda combinada ----------
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K-Means, k=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ------------------------------
# Linha 2: Séries temporais
# ------------------------------
# Escala comum para dV (mantida igual em todos os clusters)
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1  # margem de 10%

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # Selecionar células do cluster
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Plot das células individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Temperatura média (achatada, mas com escala real) ----------
    ax2 = ax.twinx()

    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()

    # Normalizar temperatura para o intervalo de dV e achatar visualmente
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)

    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85, label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Definir ticks reais para o eixo da direita
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])

    # ---------- Eixos e estilo ----------
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    #ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # ---------- Legenda combinada ----------
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K-Means, k=4, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 4
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ------------------------------
# Linha 2: Séries temporais
# ------------------------------
# Escala comum para dV (mantida igual em todos os clusters)
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1  # margem de 10%

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # Selecionar células do cluster
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Plot das células individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Temperatura média (achatada, mas com escala real) ----------
    ax2 = ax.twinx()

    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()

    # Normalizar temperatura para o intervalo de dV e achatar visualmente
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)

    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85, label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Definir ticks reais para o eixo da direita
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])

    # ---------- Eixos e estilo ----------
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    #ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # ---------- Legenda combinada ----------
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K-Means, k=5, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 5
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ------------------------------
# Linha 2: Séries temporais
# ------------------------------
# Escala comum para dV (mantida igual em todos os clusters)
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1  # margem de 10%

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # Selecionar células do cluster
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Plot das células individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Temperatura média (achatada, mas com escala real) ----------
    ax2 = ax.twinx()

    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()

    # Normalizar temperatura para o intervalo de dV e achatar visualmente
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)

    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85, label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Definir ticks reais para o eixo da direita
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])

    # ---------- Eixos e estilo ----------
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    #ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # ---------- Legenda combinada ----------
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Nível da albufeira

### Hierárquico (ligação média) + DTW, k=auto=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols,
        var_name='date',
        value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Nível da albufeira
# ==============================
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel['nivel_smooth'] = df_nivel['nivel']  # sem suavização

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

condensed = squareform(dist_matrix)
Z = linkage(condensed, method='average')

distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Título e legenda
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais (dV + nível) ---
global_dV_min = agg_pivot.min().min()
global_dV_max = agg_pivot.max().max()
dV_margin = (global_dV_max - global_dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais em cinzento
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid],
                color='lightgray', alpha=0.6)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color,
            linewidth=2.5, label=f'Média Cluster {cluster_id}')

    # Nível da albufeira (eixo secundário)
    ax2 = ax.twinx()
    nivel_min, nivel_max = df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (global_dV_max - global_dV_min) * 0.2 + (global_dV_max - (global_dV_max - global_dV_min) * 0.25)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2, alpha=0.8,
             label='Nível da albufeira (m)')

    # Eixo do nível
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')
    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (global_dV_max - global_dV_min) * 0.2 + (global_dV_max - (global_dV_max - global_dV_min) * 0.25)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{v:.0f}" for v in nivel_ticks_real])

    # Limites verticais coerentes
    ax.set_ylim(global_dV_min - dV_margin, global_dV_max + dV_margin)
    ax2.set_ylim(global_dV_min - dV_margin, global_dV_max + dV_margin)

    # Estilo e legendas
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais (se ainda não tiveres)
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """
    Determina a cor do ramo com base nos clusters das leaves que ele conecta.
    """
    if max(y_coords) > cut_distance:
        return "#A0A0A0"

    # Aproxima leaves conectadas
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]

    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    else:
        return "#A0A0A0"

# ---------------------------------------------------
# Figura 1: Dendrograma + Mapa
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.01)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)

leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round')

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")
ax_dendro.legend(fontsize=10)

ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors[cluster_id])
    
ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# Figura 2: Séries médias por cluster com nível
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(sorted(cluster_df['cluster'].unique())):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Nível da albufeira (escala visual)
    ax2 = ax.twinx()
    nivel_min, nivel_max = df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.2 + (dV_max - (dV_max - dV_min) * 0.25)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2, alpha=0.8, zorder=10,
             label='Nível da albufeira (m)')
    
    ax2.set_ylabel("Nível da albufeira (m)", fontsize=11)

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)

    # Ticks reais para o eixo do nível
    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.2 + (dV_max - (dV_max - dV_min) * 0.25)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{v:.1f}" for v in nivel_ticks_real])

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### Hierárquico (ligação completa) + DTW, k=auto=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols,
        var_name='date',
        value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Nível da albufeira
# ==============================
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel['nivel_smooth'] = df_nivel['nivel']  # sem suavização

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

condensed = squareform(dist_matrix)
Z = linkage(condensed, method='complete')

distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Título e legenda
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação completa.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais (dV + nível) ---
global_dV_min = agg_pivot.min().min()
global_dV_max = agg_pivot.max().max()
dV_margin = (global_dV_max - global_dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais em cinzento
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid],
                color='lightgray', alpha=0.6)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color,
            linewidth=2.5, label=f'Média Cluster {cluster_id}')

    # Nível da albufeira (eixo secundário)
    ax2 = ax.twinx()
    nivel_min, nivel_max = df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (global_dV_max - global_dV_min) * 0.2 + (global_dV_max - (global_dV_max - global_dV_min) * 0.25)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2, alpha=0.8,
             label='Nível da albufeira (m)')

    # Eixo do nível
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')
    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (global_dV_max - global_dV_min) * 0.2 + (global_dV_max - (global_dV_max - global_dV_min) * 0.25)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{v:.0f}" for v in nivel_ticks_real])

    # Limites verticais coerentes
    ax.set_ylim(global_dV_min - dV_margin, global_dV_max + dV_margin)
    ax2.set_ylim(global_dV_min - dV_margin, global_dV_max + dV_margin)

    # Estilo e legendas
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais (se ainda não tiveres)
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """
    Determina a cor do ramo com base nos clusters das leaves que ele conecta.
    """
    if max(y_coords) > cut_distance:
        return "#A0A0A0"

    # Aproxima leaves conectadas
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]

    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    else:
        return "#A0A0A0"

# ---------------------------------------------------
# Figura 1: Dendrograma + Mapa
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.01)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)

leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round')

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")
ax_dendro.legend(fontsize=10)

ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors[cluster_id])
    
ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação completa.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# Figura 2: Séries médias por cluster com nível
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(sorted(cluster_df['cluster'].unique())):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Nível da albufeira (escala visual)
    ax2 = ax.twinx()
    nivel_min, nivel_max = df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.2 + (dV_max - (dV_max - dV_min) * 0.25)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2, alpha=0.8, zorder=10,
             label='Nível da albufeira (m)')
    
    ax2.set_ylabel("Nível da albufeira (m)", fontsize=11)

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)

    # Ticks reais para o eixo do nível
    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.2 + (dV_max - (dV_max - dV_min) * 0.25)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{v:.1f}" for v in nivel_ticks_real])

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### K-Means, k=2, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 9. Carregar temperatura e nível
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window, polyorder=2)

# ==============================
# 10. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 11. Figura única: mapa + clusters + nível
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais com nível ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

nivel_min = df_nivel['nivel_smooth'].min()
nivel_max = df_nivel['nivel_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    ax2 = ax.twinx()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2.2, alpha=0.85, label='Nível da albufeira (m)')
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in nivel_ticks_real])

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### K-Means, k=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 9. Carregar temperatura e nível
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window, polyorder=2)

# ==============================
# 10. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 11. Figura única: mapa + clusters + nível
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais com nível ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

nivel_min = df_nivel['nivel_smooth'].min()
nivel_max = df_nivel['nivel_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    ax2 = ax.twinx()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2.2, alpha=0.85, label='Nível da albufeira (m)')
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in nivel_ticks_real])

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Precipitação total

### Hierárquico (ligação média) + DTW, k=auto=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")  # teu arquivo de precipitação
df_prec['data'] = pd.to_datetime(df_prec['data'])
# Não precisa de suavização, usaremos valores brutos


# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Título e legenda
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# -----------------------------
# Séries temporais médias com precipitação
# -----------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Precipitação total (barra) como eixo secundário
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    # Limites e formatação eixo principal
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')


plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais se ainda não existir
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))
clusters_present = sorted(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """Determina a cor de cada ramo do dendrograma conforme os clusters."""
    if max(y_coords) > cut_distance:
        return "#A0A0A0"  # acima do corte
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]
    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    return "#A0A0A0"

# ---------------------------------------------------
# FIGURA 1: dendrograma + mapa
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.01)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)
leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round', zorder=3)

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW', zorder=2)
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")
ax_dendro.legend(fontsize=10)

# Colorir labels das folhas
ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors[cluster_id])

ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()

plt.tight_layout()
plt.show()


# ---------------------------------------------------
# FIGURA 2: Séries médias por cluster + precipitação
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(clusters_present):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cluster_cells) == 0:
        ax.set_title(f'Cluster {cluster_id} - 0 células')
        continue

    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.8,
            label=f'Média Cluster {cluster_id}')

    # Precipitação (eixo secundário)
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Eixos e título
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()



### K-Means, k=2, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K-Means, k=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K-Means, k=4, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 4
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K-Means, k=5, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 5
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Precipitação total acumulada

### Hierárquico (ligação média) + DTW, k=auto=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")  # teu arquivo de precipitação
df_prec['data'] = pd.to_datetime(df_prec['data'])
# Não precisa de suavização, usaremos valores brutos


# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Título e legenda
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# -----------------------------
# Séries temporais médias com precipitação acumulada e mensal
# -----------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Ordenar e preparar precipitação
df_prec = df_prec.sort_values('data')
df_prec['prec_acum'] = df_prec['prec'].cumsum()

# Precipitação mensal
df_prec_monthly = df_prec.set_index('data').resample('M')['prec'].sum().reset_index()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais de dV
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Eixo secundário para precipitação
    ax2 = ax.twinx()
    # Linha acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'], color='blue', linewidth=2, alpha=0.7,
             label='Precipitação acumulada (mm)')
    # Barras mensais
    ax2.bar(df_prec_monthly['data'], df_prec_monthly['prec'], width=20, color='lightblue', alpha=0.5,
            label='Precipitação mensal (mm)')

    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2.set_ylim(0, max(df_prec['prec_acum'].max(), df_prec_monthly['prec'].max()*1.2))

    # Limites e formatação eixo principal
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais (se ainda não tiveres)
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """
    Determina a cor do ramo com base nos clusters das leaves que ele conecta.
    """
    if max(y_coords) > cut_distance:
        return "#A0A0A0"

    # Aproxima leaves conectadas
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]

    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    else:
        return "#A0A0A0"

# ---------------------------------------------------
# Figura 1: Dendrograma + Mapa
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.01)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)

leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round')

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")
ax_dendro.legend(fontsize=10)

ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors[cluster_id])

ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# Figura 2: Séries médias por cluster com precipitação acumulada
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec = df_prec.sort_values('data')
df_prec['prec_acum'] = df_prec['prec'].cumsum()

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(sorted(cluster_df['cluster'].unique())):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Precipitação acumulada (linha) no eixo secundário
    ax2 = ax.twinx()
    ax2.plot(df_prec['data'], df_prec['prec_acum'], color='blue', linewidth=2, alpha=0.8,
             label='Precipitação acumulada (mm)')
    ax2.set_ylabel("Precipitação acumulada (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # Escala real da precipitação acumulada
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # Limites e formatação eixo principal
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### K-Means, k=2, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec['prec_acum'] = df_prec['prec'].cumsum()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # --- dV ---
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação: barras + linha acumulada ---
    ax2 = ax.twinx()

    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'],
            width=20, color='royalblue', alpha=0.35, label='Precipitação mensal (mm)')

    # Linha de precipitação acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'],
             color='navy', linewidth=2.2, label='Precipitação acumulada (mm)')

    # Eixos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Limites ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # --- Estilo ---
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K-Means, k=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec['prec_acum'] = df_prec['prec'].cumsum()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # --- dV ---
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação: barras + linha acumulada ---
    ax2 = ax.twinx()

    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'],
            width=20, color='royalblue', alpha=0.35, label='Precipitação mensal (mm)')

    # Linha de precipitação acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'],
             color='navy', linewidth=2.2, label='Precipitação acumulada (mm)')

    # Eixos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Limites ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # --- Estilo ---
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Precipitação total anual acumulada

### Hierárquico (ligação média) + DTW, k=auto=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
import matplotlib.dates as mdates
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Pontos centrais
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Agrupar dV/dH
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(lambda x: x.year+1 if x.month>=10 else x.year)
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Matriz de distâncias DTW
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])
for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

condensed = squareform(dist_matrix)
Z = linkage(condensed, method='average')
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte DTW+Hierárquico em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Título e legenda
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# Séries temporais médias + precipitação
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    ax2 = ax.twinx()
    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='deepskyblue', alpha=0.5, label='Precipitação mensal')
    # Linha acumulada anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'], color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    by_label = dict(zip(labels1 + labels2, lines1 + lines2))
    ax.legend(by_label.values(), by_label.keys(), fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais (se ainda não tiveres)
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    if max(y_coords) > cut_distance:
        return "#A0A0A0"

    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]

    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    else:
        return "#A0A0A0"

# ---------------------------------------------------
# Figura 1: Dendrograma + Mapa
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.01)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)

leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round')

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")
ax_dendro.legend(fontsize=10)

ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors[cluster_id])

ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# Figura 2: Séries médias por cluster + precipitação
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(sorted(cluster_df['cluster'].unique())):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Precipitação mensal (barras) + acumulada anual (linha)
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='deepskyblue', alpha=0.5, label='Precipitação mensal')

    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'], color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    by_label = dict(zip(labels1 + labels2, lines1 + lines2))
    ax.legend(by_label.values(), by_label.keys(), fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### k=2, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação (acumulado anual)
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(
    lambda x: x.year if x.month < 10 else x.year + 1
)

# Calcular acumulado dentro de cada ano hidrológico
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters (com precipitação anual acumulada)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# Linha 2: Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Linhas cinzentas individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Precipitação ----------
    ax2 = ax.twinx()

    # Barras = precipitação mensal
    ax2.bar(df_prec['data'], df_prec['prec'], color='deepskyblue', alpha=0.5,
            width=15, label='Precipitação (mm)')

    # Linha = acumulado anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'],
                 color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    # Eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

### k=3, grid = 100 m

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação (acumulado anual)
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(
    lambda x: x.year if x.month < 10 else x.year + 1
)

# Calcular acumulado dentro de cada ano hidrológico
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters (com precipitação anual acumulada)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# Linha 2: Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Linhas cinzentas individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Precipitação ----------
    ax2 = ax.twinx()

    # Barras = precipitação mensal
    ax2.bar(df_prec['data'], df_prec['prec'], color='deepskyblue', alpha=0.5,
            width=15, label='Precipitação (mm)')

    # Linha = acumulado anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'],
                 color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    # Eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

precipitação total

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40, label='Centro de massa ASC/DESC')

for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.4, label=f'Cluster {cluster_id+1}')

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10)

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries dV
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação (barras + linha média) ---
    ax2 = ax.twinx()

    # Barras de precipitação
    bar_container = ax2.bar(
        df_prec['data'], df_prec['prec'],
        width=20, color='royalblue', alpha=0.35, label='Precipitação (mm)'
    )

    # Coordenadas dos centros das barras e respetivos valores
    bar_centers = [bar.get_x() + bar.get_width()/2 for bar in bar_container]
    bar_heights = [bar.get_height() for bar in bar_container]

    # Linha conectando os pontos médios das barras
    ax2.plot(bar_centers, bar_heights, color='navy', linewidth=2.0, label='Tendência da precipitação')

    # Eixos e rótulos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Eixos e estilo ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

precipitação acumulada total

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40, label='Centro de massa ASC/DESC')

for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.4, label=f'Cluster {cluster_id+1}')

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa de Clusters de dV com Centro de Células ASC/DESC", fontsize=16)
ax_map.set_axis_off()
ax_map.legend(fontsize=10)

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec['prec_acum'] = df_prec['prec'].cumsum()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # --- dV ---
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação: barras + linha acumulada ---
    ax2 = ax.twinx()

    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'],
            width=20, color='royalblue', alpha=0.35, label='Precipitação mensal (mm)')

    # Linha de precipitação acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'],
             color='navy', linewidth=2.2, label='Precipitação acumulada (mm)')

    # Eixos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Limites ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # --- Estilo ---
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


precipitação acumulada por ano

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação (acumulado anual)
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(
    lambda x: x.year if x.month < 10 else x.year + 1
)

# Calcular acumulado dentro de cada ano hidrológico
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters (com precipitação anual acumulada)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: Mapa
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(ax=ax_map,
                                                               color=cluster_colors[int(row['cluster'])],
                                                               alpha=0.4)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40, label='Centro de massa ASC/DESC')

for cluster_id in clusters_present:
    ax_map.scatter([], [], color=cluster_colors[cluster_id], alpha=0.4, label=f'Cluster {cluster_id+1}')

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa de Clusters de dV com Centro de Células ASC/DESC", fontsize=16)
ax_map.set_axis_off()
ax_map.legend(fontsize=10)

# Linha 2: Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Linhas cinzentas individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Precipitação ----------
    ax2 = ax.twinx()

    # Barras = precipitação mensal
    ax2.bar(df_prec['data'], df_prec['prec'], color='deepskyblue', alpha=0.5,
            width=15, label='Precipitação (mm)')

    # Linha = acumulado anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'],
                 color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    # Eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()
